# 01 — Sensibilidade dos thresholds — Qwen3.5-4B

Notebook extraído do experimento  do Qwen3.5-4B.

Contém somente as duas análises de sensibilidade realizadas no estudo:

1. sensibilidade do threshold de **Coverage**, mantendo o matching fixo em 0,75;
2. sensibilidade do threshold de **matching SBERT**, mantendo Coverage fixo em 0,75.

Os thresholds avaliados são 0,60, 0,65, 0,70, 0,75 e 0,80.

O notebook reaproveita as matrizes de similaridade `.npz` produzidas pelo experimento few-shot do Qwen3.5-4B e não executa novamente o modelo Qwen.

## 1. Sensibilidade ao threshold de Coverage

In [ ]:
# ============================================================
# Sensibilidade ao threshold de Coverage — Qwen3.5-4B
# ============================================================
import math
import gzip
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# CAMINHOS E CONFIGURAÇÕES
# ------------------------------------------------------------
current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src").exists()), current)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_MODEL_DIR = PROJECT_ROOT / "results" / "qwen_few_shot" / "qwen3_5_4b"
OUTPUT_DIR = PROJECT_ROOT / "results" / "additional_analysis" / "threshold_sensitivity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAMINHO_PERFIS_DOC = DATA_DIR / "perfis_documento_qwen.json"
DIR_SIM_MATRICES = RESULTS_MODEL_DIR / "sim_matrices"

CAMINHO_CACHE_COVERAGE = OUTPUT_DIR / "cache_sensibilidade_coverage_qwen3_5_4B.pkl.gz"
CAMINHO_SENS_COVERAGE = OUTPUT_DIR / "sensibilidade_threshold_coverage_qwen3_5_4B.csv"

SBERT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
BATCH_SBERT = 256
THRESHOLD_MATCHING_FIXO = 0.75
THRESHOLD_RELEVANCIA = 2
LIMIAR_REJEICAO = 0.0
THRESHOLDS_COVERAGE = [0.60, 0.65, 0.70, 0.75, 0.80]
RECRIAR_CACHE_COVERAGE = False

# ------------------------------------------------------------
# VALIDAÇÃO DE ENTRADAS
# ------------------------------------------------------------
if not CAMINHO_PERFIS_DOC.exists():
    raise FileNotFoundError(CAMINHO_PERFIS_DOC)
if not DIR_SIM_MATRICES.exists():
    raise FileNotFoundError(DIR_SIM_MATRICES)

arquivos_npz = sorted(DIR_SIM_MATRICES.glob("*.npz"))
if not arquivos_npz:
    raise RuntimeError(f"Nenhuma matriz .npz encontrada em {DIR_SIM_MATRICES}")

autores_npz = [p.stem for p in arquivos_npz]
print(f"Matrizes encontradas: {len(arquivos_npz)} autores")

# ============================================================
# FUNÇÕES DE MATCHING E MÉTRICAS
# ============================================================
def matching_greedy_global(sim, gold_weights, theta):
    K, G = sim.shape
    mw = np.zeros(K, dtype=np.int32)
    mi = np.full(K, -1, dtype=np.int32)
    ms = np.max(sim, axis=1).astype(np.float32) if G > 0 else np.zeros(K, dtype=np.float32)

    if K == 0 or G == 0:
        return mw, mi, ms

    ordem = np.argsort(-sim, axis=None, kind="stable")
    pred_usada = np.zeros(K, dtype=bool)
    gold_usado = np.zeros(G, dtype=bool)

    for flat_idx in ordem:
        i, j = np.unravel_index(int(flat_idx), sim.shape)
        score = float(sim[i, j])

        if score < theta or score <= LIMIAR_REJEICAO: break
        if pred_usada[i] or gold_usado[j]: continue

        mw[i], mi[i], ms[i] = int(gold_weights[j]), int(j), score
        pred_usada[i], gold_usado[j] = True, True
    return mw, mi, ms

def dcg_at_k(rels, k):
    return sum((2 ** int(rel) - 1) / math.log2(i + 1) for i, rel in enumerate(rels[:k], start=1) if rel > 0)

def ndcg_at_k(mw, gold_weights, k):
    idcg = dcg_at_k(np.sort(gold_weights)[::-1], k)
    return dcg_at_k(mw, k) / idcg if idcg > 0 else 0.0

def precision_at_k(mw, k):
    return float(np.sum(mw[:k] >= THRESHOLD_RELEVANCIA)) / k if k > 0 else 0.0

def recall_at_k(mw, gold_weights, k):
    total_rel = int(np.sum(gold_weights >= THRESHOLD_RELEVANCIA))
    return float(np.sum(mw[:k] >= THRESHOLD_RELEVANCIA)) / total_rel if total_rel > 0 else 0.0

def average_precision_at_k(mw, gold_weights, k):
    total_rel = int(np.sum(gold_weights >= THRESHOLD_RELEVANCIA))
    if total_rel == 0: return 0.0
    hits, soma = 0, 0.0
    for i in range(min(k, len(mw))):
        if mw[i] >= THRESHOLD_RELEVANCIA:
            hits += 1
            soma += hits / (i + 1)
    return soma / total_rel

def diversity_at_k(pred_emb, k):
    X = pred_emb[:k]
    n = X.shape[0]
    if n < 2: return 0.0
    S = X @ X.T
    return 1.0 - float(S.sum() - np.trace(S)) / (n * (n - 1))

def metricas_matching_fixo(sim, gold_weights):
    mw, _, _ = matching_greedy_global(sim, gold_weights, THRESHOLD_MATCHING_FIXO)
    return {
        "nDCG@10": ndcg_at_k(mw, gold_weights, 10),
        "nDCG@20": ndcg_at_k(mw, gold_weights, 20),
        "P@5": precision_at_k(mw, 5),
        "P@10": precision_at_k(mw, 10),
        "P@20": precision_at_k(mw, 20),
        "R@5": recall_at_k(mw, gold_weights, 5),
        "R@10": recall_at_k(mw, gold_weights, 10),
        "R@20": recall_at_k(mw, gold_weights, 20),
        "MAP@10": average_precision_at_k(mw, gold_weights, 10),
        "MAP@20": average_precision_at_k(mw, gold_weights, 20),
        "Match_Valido@10": float(np.sum(mw[:10] >= 1)),
        "Match_Valido@20": float(np.sum(mw[:20] >= 1)),
    }

def coverage_por_score(scores, theta):
    scores = np.asarray(scores, dtype=np.float32)
    if scores.size == 0: return 0.0
    return float(np.mean(np.isposinf(scores) | (scores >= theta)))

# ============================================================
# CARREGA OU CRIA CACHE DO COVERAGE
# ============================================================
cache_valido = False
cache_cov = None

if CAMINHO_CACHE_COVERAGE.exists() and not RECRIAR_CACHE_COVERAGE:
    try:
        with gzip.open(CAMINHO_CACHE_COVERAGE, "rb") as f:
            cache_cov = pickle.load(f)
        autores_cache = set(cache_cov.get("autores", {}).keys())
        cache_valido = (cache_cov.get("sbert_model") == SBERT_MODEL_NAME and autores_cache == set(autores_npz))

        if cache_valido:
            print(f"Cache de Coverage reutilizado:\n{CAMINHO_CACHE_COVERAGE}")
        else:
            print("Cache existente não corresponde às matrizes atuais. O cache será reconstruído.")
    except Exception as e:
        print(f"Não foi possível reutilizar o cache:\n{e}")
        cache_cov = None

# ============================================================
# PRIMEIRA EXECUÇÃO: reconstrói cache
# ============================================================
if not cache_valido:
    print("\n>> Construindo cache de Coverage.\n>> O Qwen NÃO será executado.")
    with open(CAMINHO_PERFIS_DOC, "r", encoding="utf-8") as f:
        perfis_doc = json.load(f)

    pred_tags_por_autor, vocab = {}, set()

    for p in tqdm(arquivos_npz, desc="Lendo matrizes"):
        with np.load(p, allow_pickle=False) as z:
            pred_tags = [str(x) for x in z["pred_tags"][:20]]
        pred_tags_por_autor[p.stem] = pred_tags
        vocab.update(pred_tags)

    for autor in autores_npz:
        docs = perfis_doc.get(autor, {})
        for _, pedacos in docs.items():
            for p in pedacos:
                if isinstance(p, str) and p.strip():
                    vocab.add(p)

    vocab.discard("")
    vocab = sorted(vocab)
    print(f"\nStrings únicas a encodar para Coverage: {len(vocab):,}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f">> Carregando SBERT em {device}:\n{SBERT_MODEL_NAME}")
    modelo_sbert = SentenceTransformer(SBERT_MODEL_NAME, device=device)

    emb = modelo_sbert.encode(vocab, batch_size=BATCH_SBERT, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True, device=device)
    emb = np.asarray(emb, dtype=np.float32)
    idx = {texto: i for i, texto in enumerate(vocab)}

    autores_cache = {}
    for autor in tqdm(autores_npz, desc="Calculando scores máximos de Coverage"):
        pred_tags = pred_tags_por_autor.get(autor, [])
        docs = perfis_doc.get(autor, {})
        pred_idx = [idx[t] for t in pred_tags if t in idx]
        pred_emb = emb[pred_idx] if pred_idx else np.zeros((0, emb.shape[1]), dtype=np.float32)

        pred10, pred20 = pred_tags[:10], pred_tags[:20]
        set10, set20 = set(pred10), set(pred20)
        score10, score20 = [], []

        for _, pedacos in docs.items():
            pedacos_unicos = set(p for p in pedacos if isinstance(p, str) and p.strip())
            if not pedacos_unicos or pred_emb.shape[0] == 0:
                score10.append(-np.inf)
                score20.append(-np.inf)
                continue

            exato10, exato20 = bool(set10 & pedacos_unicos), bool(set20 & pedacos_unicos)
            s10, s20 = (np.inf if exato10 else -np.inf), (np.inf if exato20 else -np.inf)

            if not (exato10 and exato20):
                pidx = [idx[p] for p in pedacos_unicos if p in idx]
                if pidx:
                    sim = pred_emb[:20] @ emb[pidx].T
                    if not exato10 and sim.shape[0] > 0:
                        s10 = float(np.max(sim[:min(10, sim.shape[0]), :]))
                    if not exato20 and sim.shape[0] > 0:
                        s20 = float(np.max(sim))

            score10.append(s10)
            score20.append(s20)

        autores_cache[autor] = {
            "score10": np.asarray(score10, dtype=np.float32),
            "score20": np.asarray(score20, dtype=np.float32),
            "D@10": diversity_at_k(pred_emb, 10),
            "D@20": diversity_at_k(pred_emb, 20),
        }

    cache_cov = {"sbert_model": SBERT_MODEL_NAME, "autores": autores_cache}
    with gzip.open(CAMINHO_CACHE_COVERAGE, "wb") as f:
        pickle.dump(cache_cov, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"\nCache salvo em:\n{CAMINHO_CACHE_COVERAGE}")
    del modelo_sbert, emb, idx, vocab
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# ============================================================
# MÉTRICAS FIXAS DO MATCHING
# ============================================================
fixas = defaultdict(list)
for p in tqdm(arquivos_npz, desc="Métricas fixas do matching"):
    with np.load(p, allow_pickle=False) as z:
        sim, gold_weights = z["sim"].astype(np.float32), z["gold_weights"].astype(np.int32)

    for nome, valor in metricas_matching_fixo(sim, gold_weights).items():
        fixas[nome].append(float(valor))

medias_fixas = {k: float(np.mean(v)) if v else 0.0 for k, v in fixas.items()}
d10 = float(np.mean([cache_cov["autores"][a]["D@10"] for a in autores_npz]))
d20 = float(np.mean([cache_cov["autores"][a]["D@20"] for a in autores_npz]))

# ============================================================
# SENSIBILIDADE DE COVERAGE E SALVAMENTO DO CSV
# ============================================================
linhas = []
for theta_cov in THRESHOLDS_COVERAGE:
    cov10 = [coverage_por_score(cache_cov["autores"][a]["score10"], theta_cov) for a in autores_npz]
    cov20 = [coverage_por_score(cache_cov["autores"][a]["score20"], theta_cov) for a in autores_npz]

    linhas.append({
        "θ_coverage": theta_cov,
        "θ_matching": THRESHOLD_MATCHING_FIXO,
        "threshold_relevancia_P_R_MAP": THRESHOLD_RELEVANCIA,
        "C@10": float(np.mean(cov10)),
        "C@20": float(np.mean(cov20)),
        **medias_fixas,
        "D@10": d10,
        "D@20": d20,
        "N": len(autores_npz)
    })

df_sensibilidade_coverage = pd.DataFrame(linhas)
ordem = ["θ_coverage", "θ_matching", "threshold_relevancia_P_R_MAP", "C@10", "C@20", "nDCG@10", "nDCG@20", "P@5", "P@10", "P@20", "R@5", "R@10", "R@20", "MAP@10", "MAP@20", "D@10", "D@20", "Match_Valido@10", "Match_Valido@20", "N"]
df_sensibilidade_coverage = df_sensibilidade_coverage[[c for c in ordem if c in df_sensibilidade_coverage.columns]]

print("\n" + "=" * 115)
print("SENSIBILIDADE AO THRESHOLD DE COVERAGE — Qwen3.5-4B")
print(f"Threshold de matching fixo = {THRESHOLD_MATCHING_FIXO:.2f}")
print("=" * 115)
print(df_sensibilidade_coverage.round(4).to_string(index=False))

df_sensibilidade_coverage.to_csv(CAMINHO_SENS_COVERAGE, index=False, encoding="utf-8-sig")
print(f"\nSalvo em:\n{CAMINHO_SENS_COVERAGE}")

## 2. Sensibilidade ao threshold de matching SBERT

In [ ]:
# ============================================================
# Sensibilidade ao threshold de MATCHING SBERT — Qwen3.5-4B
#
# Reaproveita diretamente as matrizes .npz já salvas.
# NÃO executa Qwen.
# NÃO executa SBERT novamente.
# Coverage fica fixo em theta_cov = 0.75.
# ============================================================
import math
import gzip
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# CAMINHOS E CONFIGURAÇÕES
# ------------------------------------------------------------
DIR_SIM_MATRICES = RESULTS_MODEL_DIR / "sim_matrices"
CAMINHO_CACHE_COVERAGE = OUTPUT_DIR / "cache_sensibilidade_coverage_qwen3_5_4B.pkl.gz"
CAMINHO_SENS_MATCHING = OUTPUT_DIR / "sensibilidade_threshold_matching_qwen3_5_4B.csv"

THRESHOLD_COVERAGE_FIXO = 0.75
THRESHOLD_RELEVANCIA = 2
LIMIAR_REJEICAO = 0.0
THRESHOLDS_MATCHING = [0.60, 0.65, 0.70, 0.75, 0.80]

# ------------------------------------------------------------
# ARQUIVOS .NPZ
# ------------------------------------------------------------
if not DIR_SIM_MATRICES.exists():
    raise FileNotFoundError(DIR_SIM_MATRICES)

arquivos_npz = sorted(DIR_SIM_MATRICES.glob("*.npz"))
if not arquivos_npz:
    raise RuntimeError(f"Nenhuma matriz .npz encontrada em {DIR_SIM_MATRICES}")

autores_npz = [p.stem for p in arquivos_npz]
print(f"Matrizes encontradas: {len(arquivos_npz)} autores")

# ------------------------------------------------------------
# CACHE DE COVERAGE
# ------------------------------------------------------------
if not CAMINHO_CACHE_COVERAGE.exists():
    raise FileNotFoundError(
        f"Cache de Coverage não encontrado:\n{CAMINHO_CACHE_COVERAGE}\n\n"
        "Execute primeiro a célula de sensibilidade de Coverage."
    )

with gzip.open(CAMINHO_CACHE_COVERAGE, "rb") as f:
    cache_cov = pickle.load(f)

if set(cache_cov.get("autores", {}).keys()) != set(autores_npz):
    raise RuntimeError("O cache de Coverage não corresponde ao conjunto atual de matrizes .npz.")

# ============================================================
# FUNÇÕES DE MATCHING E MÉTRICAS
# ============================================================
def matching_greedy_global(sim, gold_weights, theta):
    K, G = sim.shape
    mw = np.zeros(K, dtype=np.int32)
    mi = np.full(K, -1, dtype=np.int32)
    ms = np.max(sim, axis=1).astype(np.float32) if G > 0 else np.zeros(K, dtype=np.float32)

    if K == 0 or G == 0: return mw, mi, ms

    ordem = np.argsort(-sim, axis=None, kind="stable")
    pred_usada = np.zeros(K, dtype=bool)
    gold_usado = np.zeros(G, dtype=bool)

    for flat_idx in ordem:
        i, j = np.unravel_index(int(flat_idx), sim.shape)
        score = float(sim[i, j])

        if score < theta or score <= LIMIAR_REJEICAO: break
        if pred_usada[i] or gold_usado[j]: continue

        mw[i], mi[i], ms[i] = int(gold_weights[j]), int(j), score
        pred_usada[i], gold_usado[j] = True, True
    return mw, mi, ms

def dcg_at_k(rels, k):
    return sum((2 ** int(rel) - 1) / math.log2(i + 1) for i, rel in enumerate(rels[:k], start=1) if rel > 0)

def ndcg_at_k(mw, gold_weights, k):
    idcg = dcg_at_k(np.sort(gold_weights)[::-1], k)
    return dcg_at_k(mw, k) / idcg if idcg > 0 else 0.0

def precision_at_k(mw, k):
    return float(np.sum(mw[:k] >= THRESHOLD_RELEVANCIA)) / k if k > 0 else 0.0

def recall_at_k(mw, gold_weights, k):
    total_rel = int(np.sum(gold_weights >= THRESHOLD_RELEVANCIA))
    return float(np.sum(mw[:k] >= THRESHOLD_RELEVANCIA)) / total_rel if total_rel > 0 else 0.0

def average_precision_at_k(mw, gold_weights, k):
    total_rel = int(np.sum(gold_weights >= THRESHOLD_RELEVANCIA))
    if total_rel == 0: return 0.0
    hits, soma = 0, 0.0
    for i in range(min(k, len(mw))):
        if mw[i] >= THRESHOLD_RELEVANCIA:
            hits += 1
            soma += hits / (i + 1)
    return soma / total_rel

def coverage_por_score(scores, theta):
    scores = np.asarray(scores, dtype=np.float32)
    if scores.size == 0: return 0.0
    return float(np.mean(np.isposinf(scores) | (scores >= theta)))

# ============================================================
# COVERAGE E DIVERSITY FIXOS (theta_cov = 0.75)
# ============================================================
coverage10_fixo = float(np.mean([coverage_por_score(cache_cov["autores"][a]["score10"], THRESHOLD_COVERAGE_FIXO) for a in autores_npz]))
coverage20_fixo = float(np.mean([coverage_por_score(cache_cov["autores"][a]["score20"], THRESHOLD_COVERAGE_FIXO) for a in autores_npz]))
div10_fixo = float(np.mean([cache_cov["autores"][a]["D@10"] for a in autores_npz]))
div20_fixo = float(np.mean([cache_cov["autores"][a]["D@20"] for a in autores_npz]))

# ============================================================
# SENSIBILIDADE AO THRESHOLD DO MATCHING
# ============================================================
linhas = []
for theta_match in THRESHOLDS_MATCHING:
    medias = defaultdict(list)
    for p in tqdm(arquivos_npz, desc=f"theta_match={theta_match:.2f}"):
        with np.load(p, allow_pickle=False) as z:
            sim = z["sim"].astype(np.float32)
            gold_weights = z["gold_weights"].astype(np.int32)

        mw, _, _ = matching_greedy_global(sim, gold_weights, theta_match)

        medias["nDCG@10"].append(ndcg_at_k(mw, gold_weights, 10))
        medias["nDCG@20"].append(ndcg_at_k(mw, gold_weights, 20))
        medias["P@5"].append(precision_at_k(mw, 5))
        medias["P@10"].append(precision_at_k(mw, 10))
        medias["P@20"].append(precision_at_k(mw, 20))
        medias["R@5"].append(recall_at_k(mw, gold_weights, 5))
        medias["R@10"].append(recall_at_k(mw, gold_weights, 10))
        medias["R@20"].append(recall_at_k(mw, gold_weights, 20))
        medias["MAP@10"].append(average_precision_at_k(mw, gold_weights, 10))
        medias["MAP@20"].append(average_precision_at_k(mw, gold_weights, 20))
        medias["Match_Valido@10"].append(float(np.sum(mw[:10] >= 1)))
        medias["Match_Valido@20"].append(float(np.sum(mw[:20] >= 1)))

    linhas.append({
        "θ_matching": theta_match,
        "θ_coverage": THRESHOLD_COVERAGE_FIXO,
        "threshold_relevancia_P_R_MAP": THRESHOLD_RELEVANCIA,
        "nDCG@10": float(np.mean(medias["nDCG@10"])),
        "nDCG@20": float(np.mean(medias["nDCG@20"])),
        "P@5": float(np.mean(medias["P@5"])),
        "P@10": float(np.mean(medias["P@10"])),
        "P@20": float(np.mean(medias["P@20"])),
        "R@5": float(np.mean(medias["R@5"])),
        "R@10": float(np.mean(medias["R@10"])),
        "R@20": float(np.mean(medias["R@20"])),
        "MAP@10": float(np.mean(medias["MAP@10"])),
        "MAP@20": float(np.mean(medias["MAP@20"])),
        "C@10": coverage10_fixo,
        "C@20": coverage20_fixo,
        "D@10": div10_fixo,
        "D@20": div20_fixo,
        "Match_Valido@10": float(np.mean(medias["Match_Valido@10"])),
        "Match_Valido@20": float(np.mean(medias["Match_Valido@20"])),
        "N": len(arquivos_npz),
    })

df_sensibilidade_matching = pd.DataFrame(linhas)
ordem = ["θ_matching", "θ_coverage", "threshold_relevancia_P_R_MAP", "nDCG@10", "nDCG@20", "P@5", "P@10", "P@20", "R@5", "R@10", "R@20", "MAP@10", "MAP@20", "C@10", "C@20", "D@10", "D@20", "Match_Valido@10", "Match_Valido@20", "N"]
df_sensibilidade_matching = df_sensibilidade_matching[[c for c in ordem if c in df_sensibilidade_matching.columns]]

print("\n" + "=" * 115)
print("SENSIBILIDADE AO THRESHOLD DE MATCHING SBERT — Qwen3.5-4B")
print(f"Threshold de Coverage fixo = {THRESHOLD_COVERAGE_FIXO:.2f}")
print("=" * 115)
print(df_sensibilidade_matching.round(4).to_string(index=False))

df_sensibilidade_matching.to_csv(CAMINHO_SENS_MATCHING, index=False, encoding="utf-8-sig")
print(f"\nSalvo em:\n{CAMINHO_SENS_MATCHING}")